 ## Gemini API 구조화된 출력을 활용한 영화 댓글 감정 분석

[Gemini API 구조화 된 출력 문서 링크](https://ai.google.dev/gemini-api/docs/structured-output?hl=ko&_gl=1*m9bi85*_up*MQ..*_ga*ODAzMTc0MDM1LjE3NDgzOTI3MTA.*_ga_P1DBVKWT6V*czE3NDgzOTI3MDkkbzEkZzAkdDE3NDgzOTI3MDkkajYwJGwwJGgxNjE4MjQyNjEz)

In [2]:
# 필요한 라이브러리 import
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Dict, Optional, Any
import enum
from tqdm import tqdm
from datetime import datetime
from pprint import pprint
import time

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-2.5-flash-lite')

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {gemini_model}")

# 클라이언트 초기화
client = genai.Client(api_key=api_key)

API 키 설정 확인: ✓
모델 확인: gemini-2.5-flash-lite


 ### 1. 구조화된 출력을 위한 스키마 정의

**스키마 설계 프로세스**

**1. 비즈니스 질문부터 시작**

**Q: 댓글로부터 무엇을 얻고 싶은가?**  
A: 여러 댓글의 감정을 자동으로 분석하고, 긍정/부정/중립을 판단하며, 그 근거를 명확히 파악하고 싶다

**2. 필요한 정보 나열**

**단일 댓글 분석 시:**
- 댓글 ID: 어떤 댓글인지 식별
- 핵심 키워드: ["최고", "만족", "추천"] 또는 ["실망", "불만", "최악"]
- 분석 근거: "긍정적 표현이 다수 포함되어 있음"
- 감정 분류: 긍정/부정/중립
- 신뢰도: 이 분석이 얼마나 확실한가? (매우높음/높음/보통/낮음)

**배치 분석 시:**
- 전체 분석 댓글 수: 총 몇 개를 분석했나?
- 개별 분석 결과들: 각 댓글의 상세 분석
- 배치 요약: "전체적으로 긍정 70%, 부정 20%, 중립 10%"

**3. 데이터 타입 결정**

| 정보 | 타입 | 이유 |
|------|------|------|
| 댓글 ID | `int` | 고유 숫자 식별자 |
| 핵심 키워드 | `List[str]` | 여러 개 키워드 추출 (최대 5개) |
| 분석 근거 | `str` | 자유로운 텍스트 설명 (최대 200자) |
| 감정 분류 | `Enum` | 3가지 중 정확히 하나 선택 |
| 신뢰도 | `Enum` | 4단계 중 하나 선택 |
| 전체 분석 수 | `int` | 배치 처리된 댓글 수 |
| 분석 결과 리스트 | `List[SentimentAnalysis]` | 각 댓글 분석을 배열로 |
| 배치 요약 | `str` | 전체 경향 요약 (최대 300자) |

In [3]:
# 감정 분류를 위한 Enum 정의
class SentimentType(str, enum.Enum):
    POSITIVE = "positive"      # 긍정적 감정
    NEGATIVE = "negative"      # 부정적 감정
    NEUTRAL = "neutral"        # 중립적 감정

# 신뢰도 등급을 위한 Enum 정의
class ConfidenceLevel(str, enum.Enum):
    VERY_HIGH = "very_high"    # 90% 이상
    HIGH = "high"              # 70-89%
    MEDIUM = "medium"          # 50-69%
    LOW = "low"                # 50% 미만

# 단일 댓글 감정 분석 결과 스키마
class SentimentAnalysis(BaseModel):
    comment_id: int = Field(description="댓글 ID")
    original_comment: str = Field(description="원본 댓글 내용")
    key_words: List[str] = Field(description="감정 판단의 핵심 키워드", max_items=5)
    reasoning: str = Field(description="감정 분류 이유", max_length=200)
    sentiment: SentimentType = Field(description="감정 분류 결과")
    confidence_level: ConfidenceLevel = Field(description="신뢰도 등급")

# 배치 분석을 위한 스키마 (배치 처리: 일정한 개수 별로 처리를 한다)
class BatchSentimentAnalysis(BaseModel):
    total_analyzed: int = Field(description="분석된 댓글 총 수")
    analysis_results: List[SentimentAnalysis] = Field(description="개별 댓글 분석 결과")
    batch_summary: str = Field(description="배치 분석 요약", max_length=300)

print("✅ 구조화된 출력 스키마 정의 완료")

✅ 구조화된 출력 스키마 정의 완료


/var/folders/mt/b5bzczgn14s85rhfsvnlr33h0000gn/T/ipykernel_44144/1978933801.py:18: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  key_words: List[str] = Field(description="감정 판단의 핵심 키워드", max_items=5)


 ### 2. 데이터 로드 및 탐색

In [5]:
# CSV 파일 로드
df = pd.read_csv('data/영화 댓글.csv')

print("데이터 기본 정보:")
print(f"- 전체 댓글 수: {len(df):,}개")
print(f"- 컬럼: {list(df.columns)}")
print(f"- 결측값: {df.isnull().sum().sum()}개")

# 댓글 길이 분석
df['comment_length'] = df['comment'].str.len()
print(f"\n댓글 길이 통계:")
print(f"- 평균 길이: {df['comment_length'].mean():.1f}자")
print(f"- 최대 길이: {df['comment_length'].max()}자")
print(f"- 최소 길이: {df['comment_length'].min()}자")

print(f"\n샘플 댓글 3개:")
for i in range(min(3, len(df))):
    print(f"{i+1}. [{df.iloc[i]['id']}] {df.iloc[i]['comment']}")

데이터 기본 정보:
- 전체 댓글 수: 45개
- 컬럼: ['id', 'comment']
- 결측값: 0개

댓글 길이 통계:
- 평균 길이: 36.0자
- 최대 길이: 125자
- 최소 길이: 4자

샘플 댓글 3개:
1. [9976970] 아 더빙.. 진짜 짜증나네요 목소리
2. [3819312] 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
3. [10265843] 너무재밓었다그래서보는것을추천한다


 ### 3. 구조화된 감정 분석 함수 정의

In [7]:
def analyze_sentiment_structured_single(comment_id: int, comment: str) -> Optional[SentimentAnalysis]:
    """단일 댓글을 구조화된 출력으로 감정 분석"""
    
    # 시스템 프롬프트 (분석 지침)
    system_instruction = """
너는 영화 댓글의 감정을 분석하는 전문가야.

분석 기준:
- positive: 영화에 대한 긍정적 평가, 만족, 추천, 좋아함을 표현하는 댓글
- negative: 영화에 대한 부정적 평가, 불만, 실망, 비판을 표현하는 댓글  
- neutral: 객관적 서술, 애매한 평가, 감정이 명확하지 않은 댓글

분석 과정:
1. 댓글의 전체적인 톤과 감정을 파악
2. 감정을 나타내는 핵심 키워드 추출 (최대 5개)
4. 신뢰도 등급 결정 (0.9이상: very_high, 0.7-0.89: high, 0.5-0.69: medium, 0.5미만: low)
5. 판단 근거를 간결하게 설명

항상 객관적이고 일관된 기준으로 분석해줘.
"""
    
    # 일반 프롬프트 (분석할 댓글 데이터)
    user_prompt = f"""
댓글 ID: {comment_id}
댓글 내용: "{comment}"
"""
    
    try:
        # Gemini 전용 config 객체 생성
        generation_config = types.GenerateContentConfig(
            temperature=0.1,  # 일관성을 위해 낮은 창의성
            top_p=0.8,
            max_output_tokens=500,
            response_mime_type='application/json',
            response_schema=SentimentAnalysis,
            system_instruction=system_instruction
        )
        
        response = client.models.generate_content(
            model=gemini_model,
            contents=user_prompt,
            config=generation_config
        )
        
        # 구조화된 객체로 파싱
        result: SentimentAnalysis = response.parsed.model_dump()
        return result.model_dump()
        
    except Exception as e:
        print(f"❌ 단일 분석 오류 (ID: {comment_id}): {e}")
        return None

def analyze_sentiment_structured_batch(batch_data: List[tuple]) -> Optional[BatchSentimentAnalysis]:
    """배치 단위로 구조화된 감정 분석"""
    
    # 시스템 프롬프트 (분석 지침)
    system_instruction = """
너는 영화 댓글의 감정을 배치 단위로 분석하는 전문가야.

분석 기준:
- positive: 영화에 대한 긍정적 평가, 만족, 추천, 좋아함을 표현하는 댓글
- negative: 영화에 대한 부정적 평가, 불만, 실망, 비판을 표현하는 댓글  
- neutral: 객관적 서술, 애매한 평가, 감정이 명확하지 않은 댓글

각 댓글에 대해 다음을 수행해:
1. 개별 댓글의 감정 분류 및 분석
2. 핵심 키워드 추출 (최대 5개)
3. 신뢰도 점수 및 등급 부여
4. 판단 근거 제시
5. 전체 배치의 감정 경향 요약

일관성 있는 기준으로 모든 댓글을 분석해줘.
"""
    
    # 일반 프롬프트 (분석할 댓글들)
    user_prompt = "다음 영화 댓글들을 분석해주세요:\n\n"
    
    for i, (comment_id, comment) in enumerate(batch_data, 1):
        user_prompt += f"{i}. [ID: {comment_id}] \"{comment}\"\n"
    
    try:
        # Gemini 전용 config 객체 생성
        generation_config = types.GenerateContentConfig(
            temperature=0.1,
            top_p=0.8,
            max_output_tokens=1500,
            response_mime_type='application/json',
            response_schema=BatchSentimentAnalysis,
            system_instruction=system_instruction
        )
        
        response = client.models.generate_content(
            model=gemini_model,
            contents=user_prompt,
            config=generation_config
        )
        
        # 구조화된 객체로 파싱
        result: BatchSentimentAnalysis = response.parsed
        return result.model_dump()
        
    except Exception as e:
        print(f"❌ 배치 분석 오류: {e}")
        return None

 ### 4. 소규모 테스트 (구조화된 출력)

In [15]:
print("구조화된 출력 테스트: 첫 3개 댓글")
print("=" * 70)

# 테스트 데이터 준비
test_batch = [(row['id'], row['comment']) for _, row in df.head(3).iterrows()]

# 배치 분석 테스트
batch_result = analyze_sentiment_structured_batch(test_batch)

if batch_result:
    print(f"📊 배치 분석 결과:")
    print(f"- 분석된 댓글 수: {batch_result['total_analyzed']}개")
    print(f"- 배치 요약: {batch_result['batch_summary']}")
    print("\n" + "=" * 50)
    
    for i, analysis in enumerate(batch_result['analysis_results'], 1):
        print(f"\n🔍 댓글 {i} 분석 결과:")
        print(f"  ID: {analysis['comment_id']}")
        print(f"  원본: \"{analysis['original_comment']}\"")
        print(f"  감정: {analysis['sentiment'].value}")
        print(f"  신뢰도: {analysis['confidence_level'].value}")
        print(f"  핵심 키워드: {', '.join(analysis['key_words'])}")
        print(f"  분석 이유: {analysis['reasoning']}")
        print("-" * 40)
else:
    print("배치 분석 실패, 단일 분석으로 전환...")
    for comment_id, comment in test_batch:
        result = analyze_sentiment_structured_single(comment_id, comment)
        if result:
            print(f"\n🔍 댓글 [{result['comment_id']}] 분석:")
            print(f"  원본: \"{result['original_comment']}\"")
            print(f"  감정: {result['sentiment'].value}")
            print(f"  신뢰도: {result['confidence_level'].value}")
            print(f"  키워드: {', '.join(result['key_words'])}")
            print(f"  분석 이유: {result['reasoning']}")

구조화된 출력 테스트: 첫 3개 댓글
📊 배치 분석 결과:
- 분석된 댓글 수: 3개
- 배치 요약: 총 3개의 댓글을 분석한 결과, 2개의 댓글은 긍정적, 1개의 댓글은 부정적으로 분석되었습니다. 긍정적인 댓글은 영화에 대한 재미와 추천 의사를 명확히 표현하고 있으며, 부정적인 댓글은 더빙에 대한 불만을 직접적으로 드러내고 있습니다.


🔍 댓글 1 분석 결과:
  ID: 9976970
  원본: "아 더빙.. 진짜 짜증나네요 목소리"
  감정: negative
  신뢰도: very_high
  핵심 키워드: 더빙, 짜증, 목소리
  분석 이유: 댓글에서 '짜증'이라는 직접적인 부정적 감정 표현과 '더빙', '목소리'라는 불만 대상이 명확하게 드러나 부정적인 감정으로 판단됩니다.
----------------------------------------

🔍 댓글 2 분석 결과:
  ID: 3819312
  원본: "흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나"
  감정: positive
  신뢰도: high
  핵심 키워드: 초딩영화, 오버연기, 가볍지 않구나
  분석 이유: 처음에는 '초딩영화'로 오해했지만, '오버연기조차 가볍지 않다'는 표현을 통해 예상과 달리 깊이 있는 연기에 대한 긍정적 평가를 내리고 있어 복합적인 감정이지만 긍정적 뉘앙스가 강합니다.
----------------------------------------

🔍 댓글 3 분석 결과:
  ID: 10265843
  원본: "너무재밓었다그래서보는것을추천한다"
  감정: positive
  신뢰도: very_high
  핵심 키워드: 재밌었다, 추천한다
  분석 이유: 댓글에서 '재밌었다'는 직접적인 긍정적 평가와 '추천한다'는 명확한 추천 의사가 표현되어 긍정적인 감정으로 판단됩니다.
----------------------------------------


 ### 5. 전체 데이터 구조화된 감정 분석

In [16]:
print("전체 댓글 구조화된 감정 분석 시작...")
print(f"총 {len(df)}개 댓글 처리 예정")

# 결과 저장을 위한 리스트
all_results = []
batch_size = 5  # API 제한 고려

# 배치 단위로 처리
for i in tqdm(range(0, len(df), batch_size), desc="구조화된 감정 분석"):
    batch_df = df.iloc[i:i+batch_size]
    batch_data = [(row['id'], row['comment']) for _, row in batch_df.iterrows()]
    
    # 배치 분석 시도
    batch_result = analyze_sentiment_structured_batch(batch_data)
    if batch_result and len(batch_result["analysis_results"]) == len(batch_data):
        # 배치 분석 성공
        for analysis in batch_result["analysis_results"]:
            # 원본 댓글 내용 추가
            original_comment = next(comment for cid, comment in batch_data if cid == analysis.comment_id)
            
            all_results.append({
                'id': analysis.comment_id,
                'comment': original_comment,
                'sentiment': analysis.sentiment.value,
                'confidence_score': analysis.confidence_score,
                'confidence_level': analysis.confidence_level.value,
                'key_words': ', '.join(analysis.key_words),
                'reasoning': analysis.reasoning
            })
    else:
        # 배치 분석 실패시 단일 분석
        print(f"\n배치 {i//batch_size + 1} 실패, 단일 분석 진행...")
        for comment_id, comment in batch_data:
            result = analyze_sentiment_structured_single(comment_id, comment)
            if result:
                all_results.append({
                    'id': result.comment_id,
                    'comment': comment,
                    'sentiment': result.sentiment.value,
                    'confidence_score': result.confidence_score,
                    'confidence_level': result.confidence_level.value,
                    'key_words': ', '.join(result.key_words),
                    'reasoning': result.reasoning
                })
    
    # API 호출 제한 고려
    time.sleep(4)

print(f"\n✅ 구조화된 분석 완료! 총 {len(all_results)}개 댓글 처리됨")

전체 댓글 구조화된 감정 분석 시작...
총 45개 댓글 처리 예정


구조화된 감정 분석:   0%|          | 0/9 [00:00<?, ?it/s]

❌ 배치 분석 오류: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 41.349621976s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemin

구조화된 감정 분석:  11%|█         | 1/9 [00:05<00:40,  5.06s/it]

❌ 배치 분석 오류: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 36.545722075s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemin

구조화된 감정 분석:  11%|█         | 1/9 [00:08<01:06,  8.37s/it]


KeyboardInterrupt: 

 ### 6. 구조화된 결과 분석 및 시각화

In [ ]:
print("전체 댓글 구조화된 감정 분석 시작...")
print(f"총 {len(df)}개 댓글 처리 예정")

# 결과 저장을 위한 리스트
all_results = []
batch_size = 5  # API 제한 고려

# 배치 단위로 처리
for i in tqdm(range(0, len(df), batch_size), desc="구조화된 감정 분석"):
    batch_df = df.iloc[i:i+batch_size]
    batch_data = [(row['id'], row['comment']) for _, row in batch_df.iterrows()]
    
    # 배치 분석 시도
    batch_result = analyze_sentiment_structured_batch(batch_data)
    
    if batch_result and len(batch_result['analysis_results']) == len(batch_data):
        # 배치 분석 성공 
        for analysis in batch_result['analysis_results']:
            all_results.append({
                'id': analysis['comment_id'],
                'comment': analysis['original_comment'],
                'sentiment': analysis['sentiment'],
                'confidence_level': analysis['confidence_level'],
                'key_words': ', '.join(analysis['key_words']),
                'reasoning': analysis['reasoning']
            })
    else:
        # 배치 분석 실패시 단일 분석
        print(f"\n배치 {i//batch_size + 1} 실패, 단일 분석 진행...")
        for comment_id, comment in batch_data:
            result = analyze_sentiment_structured_single(comment_id, comment)
            if result:
                all_results.append({
                    'id': result['comment_id'],
                    'comment': result['original_comment'],
                    'sentiment': result['sentiment'],
                    'confidence_level': result['confidence_level'],
                    'key_words': ', '.join(result['key_words']),
                    'reasoning': result['reasoning']
                })
    
    # API 호출 제한 고려
    time.sleep(1)

print(f"\n✅ 구조화된 분석 완료! 총 {len(all_results)}개 댓글 처리됨")

 ### 7. 결과 저장

In [ ]:
df = pd.DataFrame(all_results)

# 현재 날짜 및 시간 생성
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")

# 상세 결과 CSV 저장
detailed_output = f"영화댓글_구조화된감정분석_결과_{current_time}.csv"
df.to_csv(detailed_output, index=False, encoding='utf-8-sig')
print(f"💾 상세 결과 저장: {detailed_output}")


display(df)